# Forward Motors: Stage 3 EDA
**DSBA 6165 | Mam Salan Njie and Will Moore**

The question behind this project: MMR is the wholesale price guide dealers rely on. It is accurate on average but can miss badly on an individual car, and we want to predict where it will miss before the bidding starts. That makes `sellingprice` the target, `mmr` an input, and the gap between the two the thing most worth exploring.

This notebook covers the two datasets behind our proposal:

| | Auction prices (Part A) | Damage images (Part B) |
|---|---|---|
| Source | Vehicle Sales Data, Kaggle v1 (Afridi, 2024) | CarDD (Wang, Li & Wu, 2023) |
| Samples | 558,837 transactions; 548,202 after cleaning | 4,000 images; 8,740 damage instances |
| Target | `sellingprice`, continuous (regression) | 6 damage classes + box + mask per instance |
| Split | Whole sale days: 384,673 train / 77,205 val / 86,324 test | Official: 2,816 train / 810 val / 374 test |
| Benchmark | Recorded MMR: \$1,083 MAE on the test split | Published CarDD results |

The two datasets are analyzed separately, because no photo in one is linked to a sale in the other.

## Part A. Auction-price EDA

### How to run
Open in Google Colab and choose **Runtime → Run all**. Both datasets download automatically from Kaggle when they are not on the runner's Google Drive: the auction CSV (about 85 MB) and CarDD (about 3 GB, a few minutes). Each is checked against the release we analyzed before use. The notebook trains no models.

## 1. Loading the data
Vehicle Sales Data (Kaggle, version 1) records wholesale auction sales, one row per sale. The same car can appear more than once if it was sold more than once. We check the file's hash before loading so every number below comes from the same version of the data.

In [ ]:
# @title Setup: imports, data locations and switches { display-mode: "form" }
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Polygon, Rectangle
from matplotlib.ticker import StrMethodFormatter
from PIL import Image

SEED = 42  # fixes the random sample used for scatter plots
try:
    import google.colab  # noqa: F401  (only importable inside Colab)

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Auction data. Our copy is on Google Drive; anyone else gets the public Kaggle
# version 1. Either way, the file must match this hash before we use it.
CSV_PATH = Path("/content/drive/MyDrive/forward-motors/car_prices.csv")
KAGGLE_DATASET = "syedanwarafridi/vehicle-sales-data/versions/1"
CSV_SHA256 = "32ba3ce51664e6a12c0c927ed193b41e3c4743fdf18bc0317389892aed27f556"

# CarDD damage images. The authors share the official release only on request,
# so our copy lives on Drive; anyone without it gets a public Kaggle copy.
CARDD_DRIVE_ROOTS = [
    # Copied to the Colab machine's own disk earlier in the session (fastest to read).
    Path("/content/CarDD_release/CarDD_COCO"),
    # Shortcut in our Google Drive.
    Path("/content/drive/MyDrive/CarDD_release/CarDD_COCO"),
]
# Public Kaggle copy (version 1, about 3 GB). It's a third-party upload, not the authors' own,
# so the notebook checks its counts against ours before using it. Set to False to skip Part B
# instead of downloading.
CARDD_KAGGLE_DATASET = "issamjebnouni/cardd/versions/1"
DOWNLOAD_CARDD_FROM_KAGGLE = True

# In Colab, mount Google Drive so our own copies are used when they are available.
MOUNT_GOOGLE_DRIVE = IN_COLAB

# Optional: hash every CarDD image to look for byte-identical duplicates (slow).
HASH_CARDD_PIXELS = False

### Display settings

In [ ]:
# @title Display settings: colors, table and chart style { display-mode: "form" }
# One color per partition, reused by every chart in both parts of the notebook.
SPLIT_COLORS = {
    "train": "#35618F",
    "validation": "#8066A6",
    "test": "#B87925",
}
BLUE = "#35618F"
TEAL = "#387D7A"
RUST = "#AD5D49"
LIGHT_GRAY = "#C9CED3"

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
pd.set_option("display.precision", 3)

sns.set_theme(style="whitegrid", context="notebook", palette=[BLUE, TEAL, RUST])
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "figure.facecolor": "white",
        "axes.titleweight": "semibold",
        "axes.titlepad": 14,
        "axes.labelpad": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#E5E8EB",
        "grid.linewidth": 0.6,
        "legend.frameon": False,
    }
)


def format_dollar_axis(axis, which="y"):
    """Show tick labels as $12,000 instead of 12000."""
    target = axis.yaxis if which == "y" else axis.xaxis
    target.set_major_formatter(StrMethodFormatter("${x:,.0f}"))

In [ ]:
# @title Mount Google Drive and find our data copies { display-mode: "form" }
def find_cardd_root(candidates):
    """Return the first folder that holds the CarDD annotations, or None."""
    for root in candidates:
        if (root / "annotations" / "instances_train2017.json").is_file():
            return root
    return None


# Mount Drive first: until it is mounted, none of the Drive paths exist.
# If the mount fails (access declined, or an account without our files), that's fine:
# both datasets then download from Kaggle instead. drive.mount raises ValueError on failure.
if MOUNT_GOOGLE_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive

    try:
        drive.mount("/content/drive")
    except ValueError as error:
        print(f"Google Drive not mounted ({error}); using the Kaggle downloads instead.")

CARDD_ROOT = find_cardd_root(CARDD_DRIVE_ROOTS)

csv_source = "Google Drive" if CSV_PATH.is_file() else "Kaggle download"
if CARDD_ROOT:
    cardd_source = "Google Drive (our copy from the CarDD authors)"
elif DOWNLOAD_CARDD_FROM_KAGGLE:
    cardd_source = "Kaggle download of a third-party copy (about 3 GB)"
else:
    cardd_source = "nowhere: no Drive copy and DOWNLOAD_CARDD_FROM_KAGGLE is False"
print(f"Auction CSV will come from: {csv_source}")
print(f"CarDD will come from:       {cardd_source}")

In [ ]:
# @title Load the auction CSV and check it is the right version { display-mode: "form" }
def file_sha256(path):
    """Hash a file in 1 MB chunks so a large file never sits in memory at once."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def locate_auction_csv(drive_path, kaggle_dataset):
    """Use our Drive copy if it exists; otherwise download the public Kaggle version."""
    if drive_path.is_file():
        return drive_path

    import kagglehub  # preinstalled in Colab; elsewhere: pip install kagglehub

    download_dir = Path(kagglehub.dataset_download(kaggle_dataset))
    return download_dir / "car_prices.csv"


def load_auction_data(path, expected_hash):
    """Read the CSV, but only if it is the exact file this notebook was written against."""
    if not path.is_file():
        raise FileNotFoundError(f"car_prices.csv not found at {path}")
    actual_hash = file_sha256(path)
    if actual_hash != expected_hash:
        raise ValueError(
            f"{path} has SHA-256 {actual_hash}, expected {expected_hash}. "
            "This is not the Kaggle version 1 file the project uses."
        )
    return pd.read_csv(path)


csv_path = locate_auction_csv(CSV_PATH, KAGGLE_DATASET)
cars = load_auction_data(csv_path, CSV_SHA256)
print(f"Loaded {len(cars):,} auction sales with {cars.shape[1]} columns")

# VIN identifies an individual car, so we leave it out of the preview.
display(cars.drop(columns="vin").head())

In [ ]:
cars.info()

### What each column is

| Column | Meaning |
|---|---|
| `year` | Model year of the vehicle |
| `make`, `model`, `trim`, `body` | What the vehicle is |
| `transmission` | Automatic or manual (some blanks and a few bad values) |
| `vin` | Vehicle identification number; the same car can be sold more than once |
| `state` | State code |
| `condition` | Condition code from 1 to 49 (the source does not document the scale) |
| `odometer` | Mileage at sale |
| `color`, `interior` | Exterior and interior color |
| `seller` | The consignor (whoever put the car up for auction), as recorded |
| `mmr` | Manheim Market Report value, the industry price guide |
| `sellingprice` | What the car actually sold for: **our target** |
| `saledate` | Date and time of the sale |

### How each column is used
Not every column is a model input. The table below says which ones are, along with each column's type, number of distinct values and share missing. The second table summarizes the numeric columns; skewness above 1 means a long right tail.

In [ ]:
# Everything is a model input unless it is the target, an identifier, or the split date.
column_roles = {column: "input" for column in cars.columns}
column_roles["sellingprice"] = "target"
column_roles["mmr"] = "input, and the benchmark to beat"
column_roles["vin"] = "identifier (never an input)"
column_roles["saledate"] = "sale date (used to split the data)"

column_overview = pd.DataFrame(
    {
        "role": pd.Series(column_roles),
        "type": cars.dtypes.astype(str),
        "distinct_values": cars.nunique(),
        "missing_pct": (100 * cars.isna().mean()).round(2),
    }
)
display(column_overview)

numeric_summary = cars.describe().T
numeric_summary["skewness"] = cars.select_dtypes("number").skew()
display(numeric_summary.round(2))

## 2. Missing values
What is missing, and does it matter?

In [ ]:
missing = pd.DataFrame(
    {
        "missing_records": cars.isna().sum(),
        "missing_pct": (100 * cars.isna().mean()).round(2),
    }
).sort_values("missing_records", ascending=False)
display(missing)

columns_with_gaps = missing[missing["missing_records"] > 0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(columns_with_gaps.index, columns_with_gaps["missing_pct"], color=BLUE)
ax.invert_yaxis()  # biggest gap on top
ax.set_xlabel(f"Percent of auction sales with the value missing (n = {len(cars):,})")
top_gap = columns_with_gaps.index[0]
ax.set_title(
    f"{top_gap} is missing most often ({columns_with_gaps['missing_pct'].iloc[0]:.1f}% of sales)"
)
plt.tight_layout()
plt.show()

`transmission` is missing on 11.7% of records. That is too many to drop, and transmission is not central to price for most of these vehicles, so the blanks become their own category, "unknown", instead of costing us one record in nine.

The description fields (`make`, `model`, `trim`, `body`) are each missing on about 2% of records, and the next cell shows they are usually missing together. A record with no make and no model tells us almost nothing about the car, so cleaning drops it. `sellingprice`, `mmr` and `saledate` are missing on only a handful of records; without the target or the benchmark a record cannot be used or scored.

In [ ]:
# Are the description fields missing on the same sales, or on different ones?
description_columns = ["make", "model", "trim", "body"]
missing_any = cars[description_columns].isna().any(axis=1).sum()
missing_all = cars[description_columns].isna().all(axis=1).sum()

print(f"Sales missing at least one description field: {missing_any:,}")
print(f"Sales missing all four description fields:    {missing_all:,}")
print(f"Share of those missing all four at once:      {missing_all / missing_any:.1%}")

### Duplicates
Two different questions: is any row an exact copy of another, and does the same car (VIN) show up more than once? A repeated VIN could be the same car sold again, a record entered twice, or a typo in the VIN. This file can't tell us which.

In [ ]:
# An exact duplicate is a copy of a whole row. A repeated VIN is usually a real resale.
exact_duplicate_records = cars.duplicated().sum()
repeated_vin_and_date = (
    cars.dropna(subset=["vin"]).duplicated(subset=["vin", "saledate"]).sum()
)
vin_counts = cars["vin"].value_counts()

print(f"Exact duplicate rows:                      {exact_duplicate_records:,}")
print(f"Same VIN on the same sale date:            {repeated_vin_and_date:,}")
print(f"Distinct VINs:                             {len(vin_counts):,}")
print(f"VINs that appear in more than one sale:    {(vin_counts > 1).sum():,}")

## 3. The target: auction sale price
`sellingprice` is a dollar amount, so this is a regression problem with no classes. How is it distributed?

In [ ]:
price = cars["sellingprice"].dropna()

print(f"Auction sales with a price: {len(price):,}")
print(f"Mean:     ${price.mean():,.0f}")
print(f"Median:   ${price.median():,.0f}")
print(f"Skewness: {price.skew():.2f}")

percentile_levels = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
price_percentiles = price.quantile(percentile_levels)
price_percentiles.index = [f"{level:.1%} percentile" for level in percentile_levels]
display(price_percentiles.map("${:,.0f}".format).rename("auction sale price").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].hist(price, bins=80, color=BLUE, edgecolor="white")
axes[0].set_title(f"{(price < 20_000).mean():.0%} of cars sell for under $20,000")
axes[0].set_xlabel("Auction sale price ($)")
axes[0].set_ylabel("Auction sales (n)")
format_dollar_axis(axes[0], which="x")

# A log scale spreads out the tail so the bulk of the market is easier to see.
axes[1].hist(np.log10(price[price > 0]), bins=80, color=TEAL, edgecolor="white")
axes[1].set_title("The same prices on a log scale")
axes[1].set_xlabel("log10(auction sale price in $)")
axes[1].set_ylabel("Auction sales (n)")

axes[2].boxplot(price, widths=0.5)
lower_quartile, upper_quartile = price.quantile([0.25, 0.75])
axes[2].set_title(
    f"Half of all sales fall between ${lower_quartile:,.0f} and ${upper_quartile:,.0f}"
)
axes[2].set_ylabel("Auction sale price ($)")
axes[2].set_xticks([])
format_dollar_axis(axes[2])

plt.tight_layout()
plt.show()

The distribution is right-skewed: the median is \$12,100, but the 99th percentile is \$44,900 and the maximum \$230,000.

Two consequences for modeling. Average dollar error is dominated by the expensive end, so we report percentage error alongside it and break error out by price band. And a model trained on raw dollars will spend much of its capacity on the few expensive cars, so predicting log price is a reasonable alternative to test in Stage 4.

## 4. Numeric features

Before reading the charts: `condition` isn't the 1-to-5 grade dealers usually talk about. It runs from 1 to 49, and the codes come in two clumps, 1 to 5 and roughly 19 to 49, with almost nothing in between. The documentation we have doesn't say what the codes mean, so we keep them exactly as recorded. `odometer` tops out at 999,999 miles, which looks like a placeholder rather than a real reading; cleaning drops anything over 500,000 miles.

In [ ]:
numeric_labels = {
    "year": "Model year",
    "condition": "Recorded condition code",
    "odometer": "Odometer (miles)",
    "mmr": "Recorded MMR ($)",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (column, label) in zip(axes.ravel(), numeric_labels.items(), strict=True):
    ax.hist(cars[column].dropna(), bins=60, color=BLUE, edgecolor="white")
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel("Auction sales (n)")

fig.suptitle("Distributions of the numeric columns")
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = ["year", "condition", "odometer", "mmr", "sellingprice"]
correlations = cars[numeric_columns].corr()
mmr_price_correlation = correlations.loc["mmr", "sellingprice"]

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(
    correlations,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set_title(
    f"MMR and sale price move almost in lockstep (r = {mmr_price_correlation:.2f})"
)
plt.tight_layout()
plt.show()

In [ ]:
# A 20,000-sale sample keeps the scatter plots readable; 550,000 points would be a solid blob.
SCATTER_SAMPLE_SIZE = 20_000

plot_pool = cars.dropna(subset=["sellingprice", "odometer", "condition", "mmr"])
sample = plot_pool.sample(min(SCATTER_SAMPLE_SIZE, len(plot_pool)), random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(sample["odometer"], sample["sellingprice"], s=3, alpha=0.15, color=BLUE)
axes[0].set_xlabel("Odometer (miles)")
axes[0].set_title("Mileage vs price")

axes[1].scatter(sample["condition"], sample["sellingprice"], s=3, alpha=0.15, color=TEAL)
axes[1].set_xlabel("Recorded condition code")
axes[1].set_title("Condition vs price")

# Zoom to the 99.9th percentile so a few very expensive cars don't squash the plot.
axis_limit = sample[["mmr", "sellingprice"]].quantile(0.999).max()
axes[2].scatter(sample["mmr"], sample["sellingprice"], s=3, alpha=0.15, color=RUST)
axes[2].plot(
    [0, axis_limit], [0, axis_limit], "k--", linewidth=1, label="sold exactly at MMR"
)
axes[2].set_xlim(0, axis_limit)
axes[2].set_ylim(0, axis_limit)
axes[2].set_xlabel("Recorded MMR ($)")
axes[2].set_title("MMR vs price")
axes[2].legend()
format_dollar_axis(axes[2], which="x")

for ax in axes:
    ax.set_ylabel("Auction sale price ($)")
    format_dollar_axis(ax)

fig.suptitle(f"A random sample of {len(sample):,} auction sales")
plt.tight_layout()
plt.show()

In [ ]:
# Skip model years with fewer than 50 sales; a median from a handful of cars jumps around.
MIN_SALES_PER_YEAR = 50

by_year = cars.groupby("year")["sellingprice"].agg(median_price="median", sales="size")
by_year = by_year[by_year["sales"] >= MIN_SALES_PER_YEAR]

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)

axes[0].plot(by_year.index, by_year["median_price"], marker="o", color=BLUE)
axes[0].set_ylabel("Median auction sale price ($)")
axes[0].set_title("Median price by model year")
format_dollar_axis(axes[0])

axes[1].bar(by_year.index, by_year["sales"], color=TEAL)
axes[1].set_ylabel("Auction sales (n)")
axes[1].set_xlabel("Model year")
axes[1].set_title("Sales by model year")

plt.tight_layout()
plt.show()

Mileage and price move in opposite directions, but not along a straight line: price falls steeply over the first stretch of miles and then flattens. A plain linear model would need transformed features to follow that bend; a neural network or gradient boosting can pick it up on its own.

MMR and auction sale price line up closely around the diagonal (correlation about 0.98), which is what we would expect from a price guide built on auction sales. The spread around that diagonal is the opportunity in this project, and Section 6 measures it.

## 5. Categorical features
How many distinct values does each text column have, and which values are most common?

In [ ]:
categorical_columns = [
    "make",
    "model",
    "trim",
    "body",
    "transmission",
    "state",
    "color",
    "interior",
    "seller",
]

category_rows = []
for column in categorical_columns:
    counts = cars[column].value_counts()
    category_rows.append(
        {
            "column": column,
            "distinct_values": len(counts),
            "most_common": counts.index[0],
            "most_common_share_pct": round(100 * counts.iloc[0] / counts.sum(), 1),
        }
    )

category_summary = pd.DataFrame(category_rows).set_index("column")
display(category_summary.sort_values("distinct_values", ascending=False))

In [ ]:
TOP_N = 12

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, column in zip(axes.ravel(), ["make", "body", "state", "transmission"], strict=True):
    top_values = cars[column].value_counts().head(TOP_N)
    ax.barh(top_values.index, top_values.values, color=BLUE)
    ax.invert_yaxis()  # most common on top
    ax.set_title(f"Most common {column}")
    ax.set_xlabel("Auction sales (n)")

fig.suptitle(f"The {TOP_N} most common values of four text columns")
plt.tight_layout()
plt.show()

The raw labels are not clean. `body` has `Sedan` and `sedan` as separate values, and `transmission` contains body-style words (`sedan`, `Sedan`), which points to shifted columns in a few records. Is inconsistent casing limited to `body`? The next table checks every text field: how many distinct labels it has as written, and how many remain once case and stray spaces are ignored.

In [ ]:
# How many distinct labels does each column lose once case and stray spaces are ignored?
casing_rows = []
for column in categorical_columns:
    labels = cars[column].dropna().astype(str)
    labels_as_written = labels.nunique()
    labels_ignoring_case = labels.str.strip().str.lower().nunique()
    casing_rows.append(
        {
            "column": column,
            "labels_as_written": labels_as_written,
            "labels_ignoring_case": labels_ignoring_case,
            "labels_merged": labels_as_written - labels_ignoring_case,
        }
    )

casing_table = pd.DataFrame(casing_rows).set_index("column")
display(casing_table.sort_values("labels_merged", ascending=False))

Any field with labels merged has the same value spelled more than one way, and an encoder fitted on raw labels would treat those spellings as unrelated categories. This matters more than it looks because the spelling changes over time: the earlier sales mostly say `Sedan` and `SUV`, while the latest sales mostly say `sedan` and `suv`. A model trained on the early sales would see the later ones as unfamiliar body styles. So in Stage 4 every text field is lowercased and trimmed before encoding.

In [ ]:
# Both panels are ordered by number of sales, most common first, like the bar charts above.
top_makes = cars["make"].value_counts().head(10).index
top_bodies = cars["body"].value_counts().head(8).index

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(
    data=cars[cars["make"].isin(top_makes)],
    x="make",
    y="sellingprice",
    order=top_makes,
    showfliers=False,
    color=BLUE,
    ax=axes[0],
)
axes[0].set_title("Price by make (10 most common)")

sns.boxplot(
    data=cars[cars["body"].isin(top_bodies)],
    x="body",
    y="sellingprice",
    order=top_bodies,
    showfliers=False,
    color=TEAL,
    ax=axes[1],
)
axes[1].set_title("Price by body style (8 most common labels, as written)")

for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("Auction sale price ($)")
    ax.tick_params(axis="x", rotation=40)
    format_dollar_axis(ax)

plt.tight_layout()
plt.show()

The cardinality table drives the encoding plan. `body`, `transmission` and `state` are small enough to one-hot encode directly, and `make` is borderline. `model`, `trim` and `seller` have hundreds to thousands of values, far too many for one-hot encoding, so the neural network will use learned embeddings for them.

`seller` is worth keeping despite its size. Different sellers may sell different kinds of cars or price them differently, so the seller could tell us something MMR misses. Section 7 takes a first look.

## 6. MMR as a benchmark

In [ ]:
# Compare every sale with its MMR guide value. A positive error means the car sold above guide.
benchmark = cars.dropna(subset=["sellingprice", "mmr"]).copy()
benchmark["error"] = benchmark["sellingprice"] - benchmark["mmr"]
benchmark["abs_error"] = benchmark["error"].abs()
benchmark["pct_error"] = 100 * benchmark["error"] / benchmark["mmr"]

total_ratio = benchmark["sellingprice"].sum() / benchmark["mmr"].sum()

print(f"Auction sales compared:   {len(benchmark):,}")
print(f"Mean absolute error:      ${benchmark['abs_error'].mean():,.0f}")
print(f"Median absolute error:    ${benchmark['abs_error'].median():,.0f}")
print(f"Mean signed error:        ${benchmark['error'].mean():,.0f}")
print(f"Total sold / total MMR:   {total_ratio:.1%}")
print(f"Sold above MMR:           {(benchmark['error'] > 0).mean():.1%}")
print(f"Sold below MMR:           {(benchmark['error'] < 0).mean():.1%}")
print()
for threshold in [500, 1_000, 2_000, 5_000]:
    share_missed = (benchmark["abs_error"] > threshold).mean()
    print(f"Off by more than ${threshold:,}: {share_missed:.1%} of sales")

In [ ]:
# Clip to the 0.5th-99.5th percentiles so the long tails don't flatten the histograms.
# The clipped values pile up in the two edge bins.
dollar_low, dollar_high = benchmark["error"].quantile([0.005, 0.995])
pct_low, pct_high = benchmark["pct_error"].quantile([0.005, 0.995])
share_within_2000 = (benchmark["abs_error"] <= 2_000).mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].hist(
    benchmark["error"].clip(dollar_low, dollar_high),
    bins=100,
    color=BLUE,
    edgecolor="white",
)
axes[0].axvline(0, color="black", linestyle="--", linewidth=1.2)
axes[0].set_xlabel("Auction sale price minus MMR ($)")
axes[0].set_ylabel("Auction sales (n)")
axes[0].set_title(f"{share_within_2000:.0%} of sales land within $2,000 of MMR")
format_dollar_axis(axes[0], which="x")

axes[1].hist(
    benchmark["pct_error"].clip(pct_low, pct_high), bins=100, color=RUST, edgecolor="white"
)
axes[1].axvline(0, color="black", linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Sale price minus MMR, as a percent of MMR (%)")
axes[1].set_ylabel("Auction sales (n)")
axes[1].set_title("The same error as a percent of the car's MMR")

plt.tight_layout()
plt.show()

To keep the long tails from flattening the histograms, values beyond the 0.5th and 99.5th percentiles are drawn in the two edge bins.

In [ ]:
# Five equal-sized groups of cars, from cheapest to most expensive by MMR.
band_labels = ["cheapest 20%", "20–40%", "40–60%", "60–80%", "priciest 20%"]
benchmark["mmr_band"] = pd.qcut(benchmark["mmr"], q=5, labels=band_labels)

by_band = benchmark.groupby("mmr_band", observed=True).agg(
    auction_sales=("abs_error", "size"),
    mean_abs_error=("abs_error", "mean"),
    median_mmr=("mmr", "median"),
)
# Each band's average dollar error divided by its typical MMR.
by_band["error_pct_of_median_mmr"] = 100 * by_band["mean_abs_error"] / by_band["median_mmr"]
display(by_band.round(1))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].bar(by_band.index.astype(str), by_band["mean_abs_error"], color=BLUE)
axes[0].set_ylabel("Mean absolute error ($)")
axes[0].set_title(
    f"In dollars, MMR misses most on the {by_band['mean_abs_error'].idxmax()} band"
)
format_dollar_axis(axes[0])

axes[1].bar(by_band.index.astype(str), by_band["error_pct_of_median_mmr"], color=RUST)
axes[1].set_ylabel("Mean absolute error / median MMR (%)")
axes[1].set_title(
    f"Relative to value, MMR misses most on the {by_band['error_pct_of_median_mmr'].idxmax()} band"
)

for ax in axes:
    ax.set_xlabel("Cars grouped by MMR, cheapest to priciest")

plt.tight_layout()
plt.show()

Each band is one fifth of the cars, ordered from cheapest to most expensive by MMR. The right panel divides each band's average dollar error by that band's typical MMR, so it shows the error relative to the car's value.

In [ ]:
# Five equal-width ranges of the 1-49 condition code, labeled "1–10" rather than "(0.95, 10.6]".
with_condition = benchmark.dropna(subset=["condition"]).copy()
with_condition["condition_range"] = pd.cut(with_condition["condition"], bins=5)
range_labels = {
    interval: f"{int(np.ceil(interval.left))}–{int(np.floor(interval.right))}"
    for interval in with_condition["condition_range"].cat.categories
}
with_condition["condition_range"] = with_condition["condition_range"].map(range_labels)

by_condition = (
    with_condition.groupby("condition_range")["abs_error"]
    .agg(mean_abs_error="mean", auction_sales="size")
    .reindex(range_labels.values())
)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(by_condition.index, by_condition["mean_abs_error"], color=TEAL)
ax.set_xlabel("Recorded condition code")
ax.set_ylabel("Mean absolute error ($)")
ax.set_title(
    f"MMR's average miss is largest for condition codes {by_condition['mean_abs_error'].idxmax()}"
)
format_dollar_axis(ax)
plt.tight_layout()
plt.show()

display(by_condition.round(0))

### Which way does MMR miss on rough cars?
Absolute error says how far off MMR is, not which way. The chart below shows the median signed error (auction sale price minus MMR) for every condition code with at least 500 records. Below zero means cars with that code usually sell for less than their guide value.

In [ ]:
# Skip codes with fewer than 500 sales. 500 is our judgment call, not a magic number.
MIN_SALES_PER_CODE = 500

by_condition_code = (
    benchmark.dropna(subset=["condition"])
    .groupby("condition")["error"]
    .agg(median_signed_error="median", sales="size")
)
by_condition_code = by_condition_code[by_condition_code["sales"] >= MIN_SALES_PER_CODE]

# Say what the chart shows before showing it, using the numbers behind the bars.
lowest_code, highest_code = by_condition_code.index.min(), by_condition_code.index.max()
worst_code = by_condition_code["median_signed_error"].idxmin()
best_code = by_condition_code["median_signed_error"].idxmax()
print("HOW TO READ THE NEXT CHART")
print(
    "Each bar: for cars with that condition code, the typical (median) sale price minus MMR."
)
print("  Below zero (red):  those cars usually sold for LESS than MMR.")
print("  Above zero (teal): those cars usually sold for MORE than MMR.")
print()
worst_gap = abs(by_condition_code.loc[worst_code, "median_signed_error"])
best_gap = by_condition_code.loc[best_code, "median_signed_error"]
print(f"Code {worst_code:g} sold furthest UNDER MMR: typically ${worst_gap:,.0f} under.")
print(f"Code {best_code:g} sold furthest OVER MMR:  typically ${best_gap:,.0f} over.")
print(
    f"Codes shown: {lowest_code:g} to {highest_code:g}, only codes with "
    f"{MIN_SALES_PER_CODE}+ sales. The documentation we have doesn't define the codes."
)

# Red: cars with that code usually sell under their guide value. Teal: over it.
bar_colors = np.where(by_condition_code["median_signed_error"] < 0, RUST, TEAL)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(
    by_condition_code.index,
    by_condition_code["median_signed_error"],
    color=bar_colors,
    width=0.8,
)
ax.axhline(0, color="black", linewidth=1)

# Codes 1-5 form their own cluster; the source does not say what the codes mean.
ax.axvspan(0.3, 5.7, color="#E5E8EB", zorder=0)
y_bottom, y_top = ax.get_ylim()
ax.text(3, y_bottom * 0.95, "codes 1–5:\na separate\ncluster", ha="center", va="bottom")
ax.text(ax.get_xlim()[1], y_top * 0.9, "sold over MMR ", ha="right", va="top", color=TEAL)
ax.text(
    ax.get_xlim()[1], y_bottom * 0.9, "sold under MMR ", ha="right", va="bottom", color=RUST
)

ax.set_xlabel("Recorded condition code")
ax.set_ylabel("Median sale price minus MMR ($)")
ax.set_title("Median sale price minus MMR, by recorded condition code")
format_dollar_axis(ax)
plt.tight_layout()
plt.show()

**Takeaway: low condition codes sell under MMR; high codes sell over it.**

| Condition code | Typical sale vs MMR |
|---|---|
| 1 | about \$1,650 **under** |
| 20s | \$250 to \$750 **under** |
| 40s | a few hundred dollars **over** |

- The small 1–5 group shows the same low-to-high pattern on its own.
- **Careful:** this is a pattern, not proof that MMR ignores condition. Cars with different codes also differ in make, age and sale date.
- **So what:** `condition` is worth testing as a model input. Stage 4 checks whether it actually improves predictions on held-out sales.

### The main finding: MMR is right on average, but often wrong for a single car

**1. On average, the misses cancel out.**
- All cars together sold for **98.9%** of their combined MMR; the average car sold just **\$158 under**.
- That's only because cars sold above MMR offset cars sold below it.

**2. Car by car, MMR misses a lot.**
- Typical miss: **\$700** (median). Average miss: **\$1,087**.
- **61%** of cars sold more than \$500 away from MMR, **37%** more than \$1,000 away, **14%** more than \$2,000 away.

**3. The misses aren't spread evenly.**

| Group | Average miss |
|---|---|
| Cheapest fifth of cars | \$786, or **26%** of the car's value |
| Priciest fifth of cars | **\$1,526**, or 6% of the car's value |
| Condition codes 1–10 | **\$1,396** |
| Condition codes 30–39 | \$930 |

**Caveat:** the file doesn't say when each MMR value was published, so we can't be sure a buyer saw this exact number before bidding.

## 7. Error by consignor
The data says nothing about the dealer doing the buying, but it does name the company selling each car. Is MMR more reliable for some sellers than others?

In [ ]:
# Skip consignors with fewer than 200 sales; their averages bounce around too much to compare.
MIN_SALES_PER_SELLER = 200

seller_stats = benchmark.groupby("seller").agg(
    sales=("abs_error", "size"),
    mean_abs_error=("abs_error", "mean"),
    mean_signed_error=("error", "mean"),
    median_mmr=("mmr", "median"),
)
big_sellers = seller_stats[seller_stats["sales"] >= MIN_SALES_PER_SELLER]
share_of_sales = big_sellers["sales"].sum() / len(benchmark)

print(f"Consignors in the data:                 {len(seller_stats):,}")
print(f"Consignors with {MIN_SALES_PER_SELLER}+ sales:             {len(big_sellers):,}")
print(f"Share of all sales they account for:    {share_of_sales:.1%}")
print()
print(f"Lowest mean absolute error:   ${big_sellers['mean_abs_error'].min():,.0f}")
print(f"Median mean absolute error:   ${big_sellers['mean_abs_error'].median():,.0f}")
print(f"Highest mean absolute error:  ${big_sellers['mean_abs_error'].max():,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].hist(big_sellers["mean_abs_error"], bins=40, color=BLUE, edgecolor="white")
axes[0].axvline(
    benchmark["abs_error"].mean(),
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="average over all sales",
)
axes[0].set_xlabel("Mean absolute MMR error per consignor ($)")
axes[0].set_title("Mean MMR error per consignor")
axes[0].legend()
format_dollar_axis(axes[0], which="x")

axes[1].hist(big_sellers["mean_signed_error"], bins=40, color=TEAL, edgecolor="white")
axes[1].axvline(0, color="black", linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Mean sale price minus MMR per consignor ($)")
axes[1].set_title("Mean sale price minus MMR per consignor")
format_dollar_axis(axes[1], which="x")

for ax in axes:
    ax.set_ylabel(f"Consignors with {MIN_SALES_PER_SELLER}+ sales (n)")

plt.tight_layout()
plt.show()

seller_columns = ["sales", "mean_signed_error", "mean_abs_error", "median_mmr"]

print("Ten consignors whose cars sell furthest ABOVE MMR on average")
display(big_sellers.nlargest(10, "mean_signed_error")[seller_columns].round(0))

print("Ten consignors whose cars sell furthest BELOW MMR on average")
display(big_sellers.nsmallest(10, "mean_signed_error")[seller_columns].round(0))

The consignor is whoever put the car up for auction. Judging by the names, many are lenders, rental companies and fleets. There's nothing in the file about who bought it.

If MMR were equally accurate for every seller, these averages would all sit near the overall average. They don't. Among the 313 consignors with at least 200 sales (together about 70% of all sales), the average miss ranges from \$329 to \$6,961. Several sellers whose names suggest rental, fleet or finance companies, including Hertz, two Enterprise names, Mercedes-Benz USA and TD Auto Finance, sold well under MMR on average, Hertz by nearly \$7,000.

We can't say *why* from this file: it could be the kind of cars they sell, when they sell them, or how they set reserve prices. But it makes `seller` worth testing as a model input.

## 8. When were the cars sold?
Counts describe this file, not the whole used-car market.

In [ ]:
def parse_sale_dates(raw_dates):
    """Turn 'Tue Dec 16 2014 12:30:00 GMT-0800 (PST)' into the calendar date 2014-12-16."""
    # The first four words are weekday, month, day and year; the rest is time and timezone.
    calendar_part = raw_dates.str.split(" ").str[:4].str.join(" ")
    return pd.to_datetime(calendar_part, format="%a %b %d %Y", errors="coerce")


dated = cars.dropna(subset=["saledate"]).copy()
dated["sale_dt"] = parse_sale_dates(dated["saledate"])
print(f"Dates parsed:        {dated['sale_dt'].notna().sum():,}")
print(f"Dates not parseable: {dated['sale_dt'].isna().sum():,}")

dated = dated.dropna(subset=["sale_dt"])
first_sale = dated["sale_dt"].min()
last_sale = dated["sale_dt"].max()
print(
    f"Date range: {first_sale.date()} to {last_sale.date()} ({(last_sale - first_sale).days} days)"
)

In [ ]:
# A weekly median from a few dozen sales jumps around, so we only draw the line for busy weeks.
# The 1,000 cutoff is a judgment call: busy weeks have thousands of sales and quiet weeks a few
# hundred. One week in May sits close to the cutoff, so keep an eye on it.
MIN_SALES_PER_WEEK = 1_000

# pandas labels each weekly bin by its last day (a Sunday), hence "week ending".
weekly = (
    dated.set_index("sale_dt")
    .resample("W")["sellingprice"]
    .agg(sales="size", median_price="median")
)
is_busy_week = weekly["sales"] >= MIN_SALES_PER_WEEK
share_in_busy_weeks = weekly.loc[is_busy_week, "sales"].sum() / weekly["sales"].sum()

# Blank out quiet weeks instead of dropping them, so the line breaks across gaps
# rather than drawing a straight line through weeks with no data.
busy_median_price = weekly["median_price"].where(is_busy_week)
quiet_weeks = weekly[~is_busy_week & (weekly["sales"] > 0)]

fig, axes = plt.subplots(2, 1, figsize=(13, 7.5), sharex=True)

axes[0].bar(
    weekly.index, weekly["sales"], width=6, color=np.where(is_busy_week, BLUE, LIGHT_GRAY)
)
axes[0].set_ylabel("Auction sales per week")
axes[0].set_title(
    f"{share_in_busy_weeks:.0%} of all sales fall in weeks with "
    f"{MIN_SALES_PER_WEEK:,}+ sales (gray: quieter weeks)"
)

axes[1].plot(
    weekly.index,
    busy_median_price,
    marker="o",
    markersize=3,
    color=RUST,
    label=f"weeks with {MIN_SALES_PER_WEEK:,}+ sales",
)
axes[1].scatter(
    quiet_weeks.index,
    quiet_weeks["median_price"],
    s=14,
    facecolors="none",
    edgecolors="gray",
    label=f"weeks with fewer than {MIN_SALES_PER_WEEK:,} sales",
)
axes[1].set_ylabel("Median auction sale price ($)")
axes[1].set_xlabel("Week ending")
axes[1].set_title("Weekly median price (faded points: quiet weeks)")
axes[1].legend(loc="lower left")
format_dollar_axis(axes[1])

plt.tight_layout()
plt.show()

# The same story in numbers: sales per calendar month.
monthly_sales = dated.set_index("sale_dt").resample("MS").size().rename("auction_sales")
monthly_sales.index = monthly_sales.index.strftime("%Y-%m")
display(monthly_sales.to_frame().T)

**Most sales happened in two bursts.** The file says it covers January 2014 to July 2015, but there are only 207 sales before December 2014 (206 of them in January 2014). Then come about 400,000 sales from December to early March, a lull in April (1,450 sales), and a second burst from late May through June. That's how this file was collected, not what the used-car market looked like.

**Read the price line, not the faded dots.** Weekly median prices from busy weeks sit mostly between \$11,000 and \$13,000. The wild swings, from about \$500 to \$20,000, come from weeks with only a handful of sales. The big dip in mid-May is a busy week, though, right at the start of the second burst, and it shows up again in the next chart.

In [ ]:
# Same busy-week rule as above; quiet weeks are blanked so the lines break across gaps.
dated_benchmark = benchmark.assign(sale_dt=parse_sale_dates(benchmark["saledate"]))
dated_benchmark = dated_benchmark.dropna(subset=["sale_dt"])

weekly_error = (
    dated_benchmark.set_index("sale_dt")
    .resample("W")
    .agg(
        sales=("error", "size"),
        mean_abs_error=("abs_error", "mean"),
        median_signed_error=("error", "median"),
    )
)
weekly_error = weekly_error.where(weekly_error["sales"] >= MIN_SALES_PER_WEEK)
# Start at the first busy period; early 2014 has no week with 1,000+ sales.
weekly_error = weekly_error.loc["2014-12-01":]

# Mark the busy week where MMR missed by the most, and list it so the reader can check it.
worst_week = weekly_error["mean_abs_error"].idxmax()
display(
    weekly_error.dropna().sort_values("mean_abs_error", ascending=False).head(3).round(0)
)

# Explain the chart with one concrete car before showing it.
busy_weeks = weekly_error.dropna()
typical_miss = busy_weeks["mean_abs_error"].drop(worst_week).median()
worst_miss = busy_weeks.loc[worst_week, "mean_abs_error"]
worst_direction = busy_weeks.loc[worst_week, "median_signed_error"]
print("HOW TO READ THE NEXT CHART")
print("Picture one car whose MMR says $10,000.")
print(
    f"  In a typical busy week, cars like it sold about ${typical_miss:,.0f} away from MMR,"
)
print(
    f"    so somewhere around ${10_000 - typical_miss:,.0f} to ${10_000 + typical_miss:,.0f}."
)
print(
    f"  In the week ending {worst_week:%b %d}, the average miss jumped to ${worst_miss:,.0f},"
)
print(f"    and the typical car sold ${abs(worst_direction):,.0f} UNDER MMR.")
print()
print("Top chart:    how FAR off MMR was each week (bigger = worse).")
print("Bottom chart: which WAY it missed (below zero = cars sold for less than MMR).")
print(
    f"The gap in April is not missing data we hid: those weeks had fewer than "
    f"{MIN_SALES_PER_WEEK:,} sales, too few to trust a weekly number."
)
print()

fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)

axes[0].plot(
    weekly_error.index, weekly_error["mean_abs_error"], marker="o", markersize=3, color=BLUE
)
axes[0].axhline(
    benchmark["abs_error"].mean(),
    color="black",
    linestyle="--",
    linewidth=1,
    label="average over all sales",
)
axes[0].annotate(
    f"week ending {worst_week:%b %d}\n({weekly_error.loc[worst_week, 'sales']:,.0f} sales)",
    xy=(worst_week, weekly_error.loc[worst_week, "mean_abs_error"]),
    xytext=(-150, -10),
    textcoords="offset points",
    va="center",
    arrowprops={"arrowstyle": "->", "color": "black"},
)
axes[0].set_ylabel("Mean absolute error ($)")
axes[0].set_title("How far MMR misses, week by week (busy weeks only)")
axes[0].legend(loc="upper left")
format_dollar_axis(axes[0])

axes[1].plot(
    weekly_error.index,
    weekly_error["median_signed_error"],
    marker="o",
    markersize=3,
    color=RUST,
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_ylabel("Median sale price minus MMR ($)")
axes[1].set_xlabel("Week ending")
axes[1].set_title("Which way MMR misses, week by week (below zero: cars sold under MMR)")
format_dollar_axis(axes[1])

plt.tight_layout()
plt.show()

Week to week, MMR's average miss mostly stays in a narrow band of roughly \$1,050 to \$1,150, with cars usually selling a little under MMR. The table above the chart lists the weeks where it missed most. The standout is the same mid-May week, where the average miss was about \$2,100 and cars sold about \$1,000 under MMR. That week has just over 1,000 sales, so it barely clears our busy-week cutoff, and its median price was unusually low (about \$6,000). It's worth a closer look in Stage 4.

A steady *average* miss doesn't tell us whether a 90% prediction range will keep covering 90% of sales, because the size of the occasional big misses can change while the average stays put. And these are MMR's misses, not our model's. So these charts don't settle research question 2; measuring the coverage and width of real prediction ranges on later sales does.

## 9. Cleaning decisions

In [ ]:
def drop_records(frame, mask, rule, audit_log):
    """Remove the records where mask is True and note how many went, and why."""
    kept = frame.loc[~mask]
    audit_log.append(
        {
            "rule": rule,
            "records_removed": len(frame) - len(kept),
            "records_remaining": len(kept),
        }
    )
    return kept


def clean_auction_data(raw):
    """Apply our cleaning rules in order and add sale_dt and age_at_sale."""
    audit_log = []
    clean = raw.copy()

    # These are our rules, not proof the dropped records are wrong.
    # Without the price or MMR, a sale can't be used or compared with the benchmark.
    clean = drop_records(
        clean,
        clean["sellingprice"].isna() | clean["mmr"].isna(),
        "missing sellingprice or mmr",
        audit_log,
    )
    clean = drop_records(
        clean,
        clean["make"].isna() & clean["model"].isna(),
        "missing both make and model",
        audit_log,
    )
    clean = drop_records(clean, clean["saledate"].isna(), "missing saledate", audit_log)

    # Values we don't believe: a car sold for $100 or less, or over 500,000 miles.
    clean = drop_records(
        clean, clean["sellingprice"] <= 100, "sale price at or below $100", audit_log
    )
    clean = drop_records(clean, clean["mmr"] <= 100, "MMR at or below $100", audit_log)
    clean = drop_records(
        clean, clean["odometer"] > 500_000, "odometer above 500,000 miles", audit_log
    )

    # Same VIN on the same date shows up more than once. We keep the first one; the file
    # can't tell us whether the others are double entries or something else.
    repeated = clean.duplicated(subset=["vin", "saledate"], keep="first")
    clean = drop_records(clean, repeated, "same VIN on the same sale date", audit_log)

    clean = clean.copy()
    clean["transmission"] = clean["transmission"].fillna("unknown")
    clean["sale_dt"] = parse_sale_dates(clean["saledate"])
    clean = drop_records(
        clean, clean["sale_dt"].isna(), "sale date not parseable", audit_log
    )

    # Age at sale; -1 is allowed because a 2015 model year can be sold in 2014.
    clean = clean.copy()
    clean["age_at_sale"] = clean["sale_dt"].dt.year - clean["year"]
    clean = drop_records(
        clean,
        ~clean["age_at_sale"].between(-1, 50),
        "age at sale outside -1 to 50 years",
        audit_log,
    )

    return clean, pd.DataFrame(audit_log)

In [ ]:
clean, cleaning_audit = clean_auction_data(cars)
display(cleaning_audit)

# Every removed record must be accounted for by exactly one rule.
assert cleaning_audit["records_removed"].sum() + len(clean) == len(cars)

print(f"Kept {len(clean):,} of {len(cars):,} auction sales ({len(clean) / len(cars):.1%}).")
display(
    clean[
        ["year", "age_at_sale", "odometer", "condition", "mmr", "sellingprice", "sale_dt"]
    ].head()
)

Each removal has a specific reason rather than a blanket `dropna()`, and the audit table reconciles to the final count: 548,202 of 558,837 records (98.1%) are kept.

Records missing the target or the benchmark cannot be used or scored. Records missing both make and model describe no identifiable vehicle. We also drop prices of \$100 or less and odometers over 500,000 miles; those are our calls about what's believable, not proof the records are wrong. When the same VIN shows up more than once on the same sale date, we keep the first record; the file can't tell us whether the others are double entries or something else. Missing `transmission` becomes "unknown".

The one engineered feature is `age_at_sale`, sale year minus model year. A 2010 model means something different in a 2014 sale than in a 2015 one, and age is what drives depreciation. An age of −1 is a next-model-year vehicle sold early.

## 10. Train, validation and test split
The split is chronological: the oldest 70% of records for training, the next 15% for validation and for calibrating the prediction range, and the newest 15% held out for testing. A random split would let the model learn from sales after the test period, information no buyer has when bidding, and would make results look better than they are.

Each sale day is kept whole, with boundaries at the day closest to 70% and 85% of records, so no day is split between two partitions.

In [ ]:
def split_by_sale_day(frame, train_share=0.70, validation_end_share=0.85):
    """Split by date into train, validation and test without cutting any sale day in two."""
    sales_per_day = frame.groupby("sale_dt").size().sort_index()
    if len(sales_per_day) < 3:
        raise ValueError("Need at least three sale days to make three partitions.")
    cumulative_sales = sales_per_day.cumsum().to_numpy()
    total_sales = len(frame)

    # Training ends on the day closest to 70% of sales, leaving at least two later days.
    train_end_position = np.argmin(
        np.abs(cumulative_sales[:-2] - train_share * total_sales)
    )

    # Validation ends on the later day closest to 85%, leaving at least one day for test.
    later_positions = np.arange(train_end_position + 1, len(sales_per_day) - 1)
    distance_to_target = np.abs(
        cumulative_sales[later_positions] - validation_end_share * total_sales
    )
    validation_end_position = later_positions[np.argmin(distance_to_target)]

    train_end = sales_per_day.index[train_end_position]
    validation_end = sales_per_day.index[validation_end_position]

    return {
        "train": frame[frame["sale_dt"] <= train_end],
        "validation": frame[
            (frame["sale_dt"] > train_end) & (frame["sale_dt"] <= validation_end)
        ],
        "test": frame[frame["sale_dt"] > validation_end],
    }


split_frames = split_by_sale_day(clean)
train = split_frames["train"]
val = split_frames["validation"]
test = split_frames["test"]

# The partitions must not overlap in time and must account for every cleaned sale.
assert train["sale_dt"].max() < val["sale_dt"].min()
assert val["sale_dt"].max() < test["sale_dt"].min()
assert len(train) + len(val) + len(test) == len(clean)

In [ ]:
split_rows = []
for split_name, frame in split_frames.items():
    split_rows.append(
        {
            "partition": split_name,
            "auction_sales": f"{len(frame):,}",
            "share": f"{len(frame) / len(clean):.1%}",
            "first_sale": frame["sale_dt"].min().date(),
            "last_sale": frame["sale_dt"].max().date(),
            "median_price": f"${frame['sellingprice'].median():,.0f}",
            "mmr_mean_abs_error": f"${(frame['sellingprice'] - frame['mmr']).abs().mean():,.0f}",
        }
    )

split_table = pd.DataFrame(split_rows).set_index("partition")
split_table.index.name = None
display(split_table)

In [ ]:
# Weekly sales stacked by partition; a week that straddles a boundary shows both colors.
weekly_by_split = pd.DataFrame(
    {
        split_name: frame.set_index("sale_dt").resample("W").size()
        for split_name, frame in split_frames.items()
    }
).fillna(0)
busy_period = weekly_by_split.loc["2014-12-01":]
early_sales = int(weekly_by_split.loc[:"2014-11-30"].to_numpy().sum())

fig, ax = plt.subplots(figsize=(13, 4.2))
stack_bottom = np.zeros(len(busy_period))
for split_name, frame in split_frames.items():
    ax.bar(
        busy_period.index,
        busy_period[split_name],
        width=6,
        bottom=stack_bottom,
        color=SPLIT_COLORS[split_name],
        label=f"{split_name} ({len(frame):,} sales)",
    )
    stack_bottom += busy_period[split_name].to_numpy()

ax.text(
    0.01,
    0.95,
    f"Not shown: {early_sales} training sales from early 2014",
    transform=ax.transAxes,
    va="top",
    color="gray",
)
ax.set_ylabel("Auction sales per week")
ax.set_xlabel("Week ending")
ax.set_title("Weekly sales by partition: train first, then validation, then test")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
# A later sale date does not mean a different car. Count test sales whose VIN also appears in
# training; these may be resales, duplicate records, or VIN errors - the file cannot tell us which.
def normalize_vins(vins):
    """Uppercase and trim VINs so the same car always matches itself."""
    return vins.dropna().astype(str).str.strip().str.upper()


training_vins = set(normalize_vins(train["vin"]))
resold_in_test = normalize_vins(test["vin"]).isin(training_vins).sum()
print(
    f"Test sales whose VIN also appears in training: {resold_in_test:,} of {len(test):,} "
    f"({resold_in_test / len(test):.1%})"
)

Training is the winter burst. Validation is the quiet spring plus the start of the May burst, and test is the rest of June and July. So the model is always judged on sales that came after the ones it learned from. Test sales are a bit pricier than training sales (median \$13,300 against \$11,900), which makes sense if later sales include newer cars.

494 test sales (0.6%) have a VIN that also shows up in training. Those could be the same car sold twice, a record entered twice, or a VIN typo; the file can't tell us which. VIN is never a model input, but the same car can still end up on both sides of the split, so we flag it as a limitation.

### Do the three partitions look alike?
Price, mileage and age for each partition on the same scale. When the three lines sit close together, the partitions contain similar cars.

In [ ]:
comparison_labels = {
    "sellingprice": "Auction sale price ($)",
    "odometer": "Odometer (miles)",
    "age_at_sale": "Age at sale (years)",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (column, label) in zip(axes, comparison_labels.items(), strict=True):
    for split_name, frame in split_frames.items():
        # Cumulative curve: for each value, the share of sales at or below it.
        values = np.sort(frame[column].dropna().to_numpy())
        share_at_or_below = np.arange(1, len(values) + 1) / len(values)
        ax.plot(values, share_at_or_below, color=SPLIT_COLORS[split_name], label=split_name)
    ax.set_xlabel(label)
    ax.set_ylabel("Share of sales at or below this value")
    ax.set_title(label)
    ax.legend()

format_dollar_axis(axes[0], which="x")
fig.suptitle("Price, mileage and age in each partition")
plt.tight_layout()
plt.show()

### Body-style labels change spelling over time
The file writes the same body style two ways: `Sedan` early on and `sedan` later. A model only knows the labels it saw in training, so to it `sedan` looks like a brand-new kind of car. The left chart shows the spelling switch; the right chart shows how many test sales that would trip up, before and after lowercasing.

In [ ]:
CASE_COLORS = {
    "Capitalized (Sedan, SUV)": BLUE,
    "lowercase (sedan, suv)": RUST,
    "missing": LIGHT_GRAY,
}


def body_label_case(labels):
    """Label each sale's body style as Capitalized, lowercase, or missing."""
    case = pd.Series("Capitalized (Sedan, SUV)", index=labels.index)
    case[labels.fillna("").str.islower()] = "lowercase (sedan, suv)"
    case[labels.isna()] = "missing"
    return case


def share_unseen_in_training(test_labels, training_labels, lowercase):
    """Share of test sales whose body label never appears in training."""
    if lowercase:
        test_labels = test_labels.str.strip().str.lower()
        training_labels = training_labels.str.strip().str.lower()
    test_labels = test_labels.dropna()
    return 100 * (~test_labels.isin(set(training_labels.dropna()))).sum() / len(test)


# The problem: which spelling each partition uses.
case_shares = (
    pd.DataFrame(
        {
            split_name: 100 * body_label_case(frame["body"]).value_counts(normalize=True)
            for split_name, frame in split_frames.items()
        }
    )
    .T.reindex(columns=list(CASE_COLORS))
    .fillna(0)
)

# Why it matters: a label never seen in training is "unknown" to the model.
unseen = pd.Series(
    {
        "as written": share_unseen_in_training(
            test["body"], train["body"], lowercase=False
        ),
        "after lowercasing": share_unseen_in_training(
            test["body"], train["body"], lowercase=True
        ),
    }
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 1]})

case_shares.plot.barh(stacked=True, color=list(CASE_COLORS.values()), width=0.7, ax=axes[0])
axes[0].invert_yaxis()  # train on top, in time order
axes[0].set_xlim(0, 100)
axes[0].set_xlabel("Percent of auction sales in the partition")
axes[0].set_ylabel("")
axes[0].set_title("1. The problem: same body styles, spelled differently over time")
axes[0].legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.25))
for segment in axes[0].containers:
    segment_labels = [f"{value:.0f}%" if value >= 5 else "" for value in segment.datavalues]
    axes[0].bar_label(
        segment,
        labels=segment_labels,
        label_type="center",
        color="white",
        fontweight="semibold",
    )

axes[1].bar(unseen.index, unseen.values, color=[RUST, TEAL], width=0.6)
for position, value in enumerate(unseen.values):
    axes[1].text(
        position,
        value + 0.5,
        f"{value:.1f}%" if value >= 0.1 else f"{value:.3f}%",
        ha="center",
    )
axes[1].set_ylabel("Test sales with a body label\nthe model never saw in training (%)")
axes[1].set_title("2. The fix: lowercase labels first")

plt.tight_layout()
plt.show()

Lesson: clean text labels before encoding them. As written, about 20% of test sales have a body label the model never saw in training; after lowercasing, almost none do. Our earlier price-model encoder kept case, so it treated `Sedan` and `sedan` as different things. In Stage 4 every text column is lowercased and trimmed before encoding.

### The benchmark to beat
Recorded MMR's miss on the test sales. Models are chosen on validation; we report test results once at the end, compared with this number.

In [ ]:
# We do not know whether MMR was published before bidding, so this is a retrospective benchmark.
test_errors = test["sellingprice"] - test["mmr"]
test_pct_error = 100 * test_errors.abs() / test["mmr"]

test_benchmark = pd.Series(
    {
        "Auction sales in the test partition": f"{len(test):,}",
        "Mean absolute error": f"${test_errors.abs().mean():,.0f}",
        "Median absolute error": f"${test_errors.abs().median():,.0f}",
        "Mean absolute error as a percent of MMR": f"{test_pct_error.mean():.1f}%",
    },
    name="Recorded MMR on test",
)
display(test_benchmark.to_frame())

## 11. What this data cannot tell us
Several boundaries shape what we can honestly claim at the end.

- **The prices are from 2014 and 2015.** The dollar amounts will not carry over to a sale today; the method does. A dealer would retrain the same pipeline on recent purchases.
- **There is no profit information.** The data records what a car sold for, not reconditioning costs or resale. We can show an estimate is accurate, not that acting on it makes money, which is why our headline measure is mispricing flagged rather than dollars earned.
- **There is no buyer.** `seller` is the consignor, so we cannot study how buyers behave.
- **No photographs are attached to these sales.** Price and damage are tested on separate datasets, so we cannot measure how much a specific dent lowers a specific car's price.
- **Nothing records how long a car took to sell,** which rules out demand forecasting.
- **Timing and codebooks are undocumented.** We do not know when MMR was published relative to bidding, or exactly what each condition code means.

## 12. Dataset at a glance

In [ ]:
auction_glance = pd.Series(
    {
        "Dataset": "Vehicle Sales Data (wholesale auction sales)",
        "Source": "Kaggle, Syed Anwar Afridi (2024), version 1",
        "Auction sales, as downloaded": f"{len(cars):,}",
        "Auction sales, after cleaning": f"{len(clean):,} ({len(clean) / len(cars):.1%} kept)",
        "Original columns": f"{cars.shape[1]}",
        "Task": "Regression (no classes)",
        "Target": "sellingprice, auction sale price in US dollars",
        "Target range": f"${clean['sellingprice'].min():,.0f} to ${clean['sellingprice'].max():,.0f}",
        "Target median": f"${clean['sellingprice'].median():,.0f}",
        "Sale dates": f"{clean['sale_dt'].min().date()} to {clean['sale_dt'].max().date()}",
        "Split": "By date, whole sale days, about 70 / 15 / 15",
        "Train / validation / test": f"{len(train):,} / {len(val):,} / {len(test):,}",
        "Benchmark to beat": f"Recorded MMR, ${test_errors.abs().mean():,.0f} mean absolute error on test",
    },
    name="value",
)
display(auction_glance.to_frame())

## 13. What this means for our three research questions
This analysis does not answer the questions; Stages 4 and 5 do. It establishes whether each is worth asking and what it will be measured against.

**Q1, price: can we predict where MMR will miss?** Worth asking. MMR is close to right on average but misses by more than \$1,000 on 36.6% of cars, and the misses line up with things in the data: price band, condition code and seller. A pattern like that is something a model can try to learn, and Stage 4 tests whether it actually can. The number to beat is recorded MMR's \$1,083 average miss on the 86,324 test sales. We'll pick models using the validation sales and report the test result once, at the end.

**Q2, range: does a range that updates stay near 90% coverage?** This data alone can't answer it. MMR's average miss is fairly steady week to week, but that doesn't tell us how the big misses behave, and it's MMR's error, not our model's. We'll answer it by measuring the coverage and width of real prediction ranges on the later sales.

**Q3, damage: can a detector find visible damage?** The auction data can't answer this; it has no photos. It does show MMR missing more on cars with low condition codes (about \$1,396 on average versus \$930 in the middle of the scale), but we don't know whether those codes reflect visible damage. And since no CarDD photo is linked to any sale here, we can't test whether damage explains MMR's misses. Part B just looks at the photos the detector will learn from.


## 14. Summary and what this means for modeling
- **Data.** 548,202 cleaned auction sales with 16 original columns. The target, `sellingprice`, is in dollars and has a long right tail, so we report errors in dollars, in percent, and by price band.
- **Benchmark.** Recorded MMR misses by \$1,083 on average on the test sales. We choose models using validation and report test results once at the end. (We've already looked at the test sales in this EDA, and we'll say so in the report.)
- **Encoding.** One-hot for small text columns; learned embeddings for `model`, `trim` and `seller`. Every text column gets lowercased and trimmed first, using training data only. `condition` stays as recorded.
- **Model choice.** Price drops off with mileage along a curve, not a straight line. A linear model would need help with that (transformed features); a neural network or gradient boosting can learn the curve directly. Which works best is for Stage 4 to show.
- **Evaluation.** Train on earlier sales, test on later ones. Prediction ranges get judged on their actual coverage and width on later sales.

**Next, Stage 4:** baseline price models on this split. MMR alone as the benchmark, gradient boosting as the classical baseline, and a neural network with embeddings as the deep learning model.

## References

Afridi, S. A. (2024). *Vehicle Sales Data* [Dataset]. Kaggle.
https://www.kaggle.com/datasets/syedanwarafridi/vehicle-sales-data

Manheim / Cox Automotive. *MMR Best Practices* and *MMR Help*.

Romano, Y., Patterson, E., & Candès, E. (2019). Conformalized quantile regression. *NeurIPS*.

Gibbs, I., & Candès, E. (2021). Adaptive conformal inference under distribution shift. *NeurIPS*.

## Part B. CarDD damage-image EDA
CarDD is the labeled photo set the damage detector will learn from. Each photo shows one or more damaged areas, and each area is labeled with one of six damage types (dent, scratch, crack, glass shatter, lamp broken, tire flat), a box around it, and an outline of its exact shape.

### Where the CarDD files come from
Our copy comes straight from the CarDD authors, who only hand out the official release after a signed request (https://cardd-ustc.github.io/). That means someone else can't download it with code. So when our copy isn't on the runner's Google Drive, the notebook downloads a public copy someone posted on Kaggle (`issamjebnouni/cardd`, about 3 GB).

The Kaggle files are laid out differently, so the next cells check that the counts match ours exactly: photos per partition, and damage areas per damage type in every partition. If any count is off, the notebook stops. That's a consistency check on counts. It can't prove every photo, box and partition assignment is identical to ours, so we treat the Kaggle copy as its own source and say which one was used.

If you run this on your own computer instead of Colab, the notebook prints exactly where the download went (under `~/.cache/kagglehub/datasets/issamjebnouni/cardd/`). Delete just that folder afterwards if you don't want to keep 3 GB of photos around; leave the rest of `~/.cache/kagglehub/` alone, since other projects may use it.

In [ ]:
# @title Find or download the CarDD photos { display-mode: "form" }
CARDD_SPLITS = ["train", "val", "test"]

# Counts from our copy of the release. The Kaggle copy has to match all of them.
CARDD_EXPECTED_PHOTOS = {"train": 2816, "val": 810, "test": 374}
CARDD_EXPECTED_DAMAGE_AREAS = {  # per damage type, per partition
    "train": {
        "dent": 1806,
        "scratch": 2560,
        "crack": 651,
        "glass shatter": 475,
        "lamp broken": 494,
        "tire flat": 225,
    },
    "val": {
        "dent": 501,
        "scratch": 728,
        "crack": 177,
        "glass shatter": 135,
        "lamp broken": 141,
        "tire flat": 62,
    },
    "test": {
        "dent": 236,
        "scratch": 307,
        "crack": 70,
        "glass shatter": 71,
        "lamp broken": 69,
        "tire flat": 32,
    },
}


def cardd_paths_from_release(root):
    """Official layout: annotations/instances_train2017.json next to a train2017/ folder."""
    annotation_paths = {
        split: root / "annotations" / f"instances_{split}2017.json"
        for split in CARDD_SPLITS
    }
    image_dirs = {split: root / f"{split}2017" for split in CARDD_SPLITS}
    return annotation_paths, image_dirs


def cardd_paths_from_kaggle(dataset):
    """Kaggle layout: train.json next to a train/ folder."""
    import kagglehub  # preinstalled in Colab; elsewhere: pip install kagglehub

    root = Path(kagglehub.dataset_download(dataset))
    annotation_paths = {split: root / f"{split}.json" for split in CARDD_SPLITS}
    image_dirs = {split: root / split for split in CARDD_SPLITS}
    return annotation_paths, image_dirs


if CARDD_ROOT is not None:
    CARDD_ANNOTATIONS, CARDD_IMAGE_DIRS = cardd_paths_from_release(CARDD_ROOT)
    print(f"Using our copy of CarDD: {CARDD_ROOT}")
elif not DOWNLOAD_CARDD_FROM_KAGGLE:
    raise RuntimeError(
        "No CarDD copy on Drive and DOWNLOAD_CARDD_FROM_KAGGLE is False. "
        "Turn the switch on in the setup cell, or stop here: Part A is complete."
    )
else:
    print(f"Downloading CarDD from Kaggle ({CARDD_KAGGLE_DATASET}, about 3 GB)...")
    CARDD_ANNOTATIONS, CARDD_IMAGE_DIRS = cardd_paths_from_kaggle(CARDD_KAGGLE_DATASET)
    print(f"Downloaded to: {CARDD_ANNOTATIONS['train'].parent}")

# Fail here, not ten cells later, if the photos are not where the layout says.
for split, image_dir in CARDD_IMAGE_DIRS.items():
    if next(image_dir.glob("*.jpg"), None) is None:
        raise FileNotFoundError(
            f"No .jpg photos found for the {split} partition in {image_dir}"
        )

### Loading the annotations
The labels come as three COCO files, one per partition. We turn them into two tables: one row per photo, and one row per labeled damage area.

In [ ]:
# @title Load the CarDD annotations and check the counts { display-mode: "form" }
def load_cardd_tables(annotation_paths):
    """Read the three COCO files into one photo table and one damage-area table."""
    image_frames = []
    annotation_frames = []
    categories = None

    for split, path in annotation_paths.items():
        if not path.is_file():
            raise FileNotFoundError(f"CarDD annotation file missing: {path}")
        coco = json.loads(path.read_text())

        # All three files must use the same six damage types with the same ids.
        split_categories = {int(item["id"]): item["name"] for item in coco["categories"]}
        if categories is None:
            categories = split_categories
        elif split_categories != categories:
            raise ValueError(
                f"The damage types in {path.name} differ from the training file."
            )

        image_frames.append(pd.DataFrame(coco["images"]).assign(split=split))
        annotation_frames.append(pd.DataFrame(coco["annotations"]).assign(split=split))

    images = pd.concat(image_frames, ignore_index=True)
    annotations = pd.concat(annotation_frames, ignore_index=True)
    annotations["damage_class"] = annotations["category_id"].map(categories)
    return images, annotations, categories


images, annotations, categories = load_cardd_tables(CARDD_ANNOTATIONS)

# Whichever copy was loaded, its counts must match ours exactly: photos per partition, and
# damage areas per type per partition. Matching counts is strong evidence it's the same
# release, though it can't prove every box is byte-for-byte identical.
if annotations["damage_class"].isna().any():
    raise ValueError("A damage area uses a damage type id that isn't in the category list.")

photos_found = images.groupby("split").size().to_dict()
if photos_found != CARDD_EXPECTED_PHOTOS:
    raise ValueError(f"Photo counts {photos_found} differ from {CARDD_EXPECTED_PHOTOS}.")

areas_found = annotations.groupby(["split", "damage_class"]).size()
for split, expected_by_type in CARDD_EXPECTED_DAMAGE_AREAS.items():
    found_by_type = areas_found[split].to_dict()
    if found_by_type != expected_by_type:
        raise ValueError(
            f"{split} damage counts {found_by_type} differ from {expected_by_type}."
        )

print("Damage types:", ", ".join(categories.values()))
print(
    f"All counts match our copy: {len(images):,} photos and {len(annotations):,} damage areas."
)

### Is the image data balanced?
CarDD is small, 4,000 photos, so before training we checked two things: how common each damage type is, and whether train, validation and test have a similar mix. We use CarDD's official split as the authors released it.

In [ ]:
split_names = {"train": "Train", "val": "Validation", "test": "Test"}

split_sizes = (
    pd.DataFrame(
        {
            "Photos": images.groupby("split").size(),
            "Damage areas": annotations.groupby("split").size(),
        }
    )
    .reindex(list(split_names))
    .rename(index=split_names)
)
split_sizes.loc["Total"] = split_sizes.sum()
display(split_sizes)

# Percent of each partition's damage areas that belong to each damage type.
class_counts = pd.crosstab(annotations["damage_class"], annotations["split"])
class_counts = class_counts.reindex(columns=list(split_names)).rename(columns=split_names)
class_mix = 100 * class_counts / class_counts.sum()
class_mix = class_mix.loc[class_counts.sum(axis=1).sort_values(ascending=False).index]
class_mix.index.name = "Damage type"
class_mix.columns.name = None
display(class_mix.round(1).astype(str) + "%")

# Simple balance check: for each damage type, the biggest gap in share between any two partitions.
largest_gaps = (class_mix.max(axis=1) - class_mix.min(axis=1)).sort_values(ascending=False)
biggest_type = largest_gaps.index[0]
print(
    f"Biggest gap between partitions: {biggest_type}, "
    f"{class_mix.loc[biggest_type].max():.1f}% vs {class_mix.loc[biggest_type].min():.1f}% "
    f"({largest_gaps.iloc[0]:.1f} percentage points)"
)

In [ ]:
class_totals = annotations["damage_class"].value_counts().sort_values()
class_shares = 100 * class_totals / class_totals.sum()
balanced_share = 100 / len(class_totals)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(
    class_shares.index,
    class_shares.values,
    color=np.where(class_shares < balanced_share, RUST, BLUE),
)
ax.axvline(
    balanced_share,
    color="black",
    linestyle="--",
    linewidth=1,
    label=f"if all six were equal ({balanced_share:.1f}% each)",
)

# Label each bar with its share and count; the white box keeps text readable over the line.
for position, (share, count) in enumerate(zip(class_shares, class_totals, strict=True)):
    ax.text(
        share + 0.5,
        position,
        f"{share:.1f}%  ({count:,})",
        va="center",
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 1},
    )

ax.set_xlabel("Percent of all damage areas")
ax.set_xlim(0, class_shares.max() * 1.25)
imbalance_ratio = class_totals.max() / class_totals.min()
ax.set_title(
    f"{class_totals.idxmax()} is {imbalance_ratio:.0f}x as common as {class_totals.idxmin()}"
)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

The three partitions have nearly the same mix of damage types: the biggest gap is scratches, 41.7% of damage areas in validation versus 39.1% in test, and every other type is closer than that. So the three partitions have a similar mix of damage types.

The dataset as a whole is lopsided, though: scratches and dents make up 70% of all damage areas, and there are about 11 scratches for every flat tire. In Stage 4 we report average precision (AP) for each damage type plus overall mAP, so the rare types can't hide behind the common ones.

### How many damage areas are in each photo?

In [ ]:
# How many damage areas does each photo have? Zero would mean nobody labeled any damage on it.
areas_per_photo = annotations.groupby(["split", "image_id"]).size().rename("damage_areas")
per_photo = images.merge(
    areas_per_photo.reset_index(),
    left_on=["split", "id"],
    right_on=["split", "image_id"],
    how="left",
    validate="one_to_one",
)
per_photo["damage_areas"] = per_photo["damage_areas"].fillna(0).astype(int)
print(f"Photos with no damage labeled at all: {(per_photo['damage_areas'] == 0).sum()}")

most_areas = per_photo["damage_areas"].max()
fig, ax = plt.subplots(figsize=(8, 3.5))
# Bins centered on whole numbers, starting at 0 so photos with no annotations would show up.
ax.hist(
    per_photo["damage_areas"],
    bins=np.arange(-0.5, most_areas + 1.5),
    color=BLUE,
    edgecolor="white",
)
ax.set_xlabel("Damage areas in the photo")
ax.set_ylabel("Photos (n)")
ax.set_title("How many damage areas each photo has")
plt.tight_layout()
plt.show()

### Are the files consistent?
Before looking at sizes, a few sanity checks: every box should fit inside its photo, every damage area should have an outline, every photo file should exist, and no photo should appear in two partitions. Each count below should be zero.

In [ ]:
def attach_box_geometry(images, annotations):
    """One row per damage area, with its box split into columns and its photo's size attached."""
    geometry = annotations.merge(
        images[["split", "id", "file_name", "width", "height"]].rename(
            columns={"id": "image_id"}
        ),
        on=["split", "image_id"],
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    if not (geometry["_merge"] == "both").all():
        raise ValueError("A damage area points to a photo that is not in its partition.")

    # COCO boxes are [x, y, width, height] in pixels.
    if not geometry["bbox"].map(len).eq(4).all():
        raise ValueError("Every box should have exactly four numbers.")
    boxes = pd.DataFrame(
        geometry["bbox"].tolist(),
        columns=["x", "y", "box_w", "box_h"],
        index=geometry.index,
    )
    return geometry.drop(columns="_merge").join(boxes)


geometry = attach_box_geometry(images, annotations)

In [ ]:
# Annotation tools often round a box one pixel past the photo's edge, so allow that much.
EDGE_TOLERANCE_PX = 1


def find_bad_boxes(geometry, tolerance=EDGE_TOLERANCE_PX):
    """True for boxes with a missing or infinite number, no size, or that stick out of the photo."""
    # Check bad numbers first: any comparison with NaN is False, so NaN would slip through.
    numbers = geometry[["x", "y", "box_w", "box_h", "width", "height"]].astype(float)
    has_bad_number = ~np.isfinite(numbers).all(axis=1)
    return has_bad_number | (
        (geometry["width"] <= 0)
        | (geometry["height"] <= 0)
        | (geometry["box_w"] <= 0)
        | (geometry["box_h"] <= 0)
        | (geometry["x"] < 0)
        | (geometry["y"] < 0)
        | (geometry["x"] + geometry["box_w"] > geometry["width"] + tolerance)
        | (geometry["y"] + geometry["box_h"] > geometry["height"] + tolerance)
    )


bad_box = find_bad_boxes(geometry)
missing_outline = geometry["segmentation"].map(
    lambda outline: outline is None or len(outline) == 0
)
missing_photo_files = sum(
    not (CARDD_IMAGE_DIRS[row.split] / row.file_name).is_file()
    for row in images.itertuples()
)
filenames = images.groupby("split")["file_name"].apply(set)

consistency_checks = pd.Series(
    {
        "boxes that are broken or stick out of their photo": int(bad_box.sum()),
        "damage areas with no outline": int(missing_outline.sum()),
        "photo files missing": missing_photo_files,
        "same filename twice in one partition": int(
            images.duplicated(["split", "file_name"]).sum()
        ),
        "filenames in both train and validation": len(
            filenames["train"] & filenames["val"]
        ),
        "filenames in both train and test": len(filenames["train"] & filenames["test"]),
        "filenames in both validation and test": len(filenames["val"] & filenames["test"]),
    },
    name="count",
)
display(consistency_checks.to_frame())

# Box sizes below use only good boxes; nothing is clipped or changed.
geometry = geometry.loc[~bad_box].copy()
geometry["box_area_pct"] = (
    100 * (geometry["box_w"] * geometry["box_h"]) / (geometry["width"] * geometry["height"])
)

### How big are the photos, and how much of the photo does each damage box cover?
A box is a rectangle around the damage, so it also covers some undamaged paint around it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(images["width"], images["height"], s=6, alpha=0.3, color=BLUE)
axes[0].set_xlabel("Photo width (pixels)")
axes[0].set_ylabel("Photo height (pixels)")
axes[0].set_title(f"{(images['width'] == 1000).mean():.0%} of photos are 1,000 pixels wide")

axes[1].hist(images["width"] / images["height"], bins=35, color=TEAL, edgecolor="white")
axes[1].set_xlabel("Width / height")
axes[1].set_ylabel("Photos (n)")
axes[1].set_title(
    f"{(images['width'] > images['height']).mean():.0%} of photos are wider than tall"
)

plt.tight_layout()
plt.show()

# Typical share of the photo that one damage box covers, by damage type.
# (The box is a rectangle, so this overstates the damaged area itself.)
box_size_order = (
    geometry.groupby("damage_class")["box_area_pct"].median().sort_values().index
)
box_size_summary = (
    geometry.groupby("damage_class")["box_area_pct"]
    .agg(median_pct="median", damage_areas="size")
    .reindex(box_size_order)
)
display(box_size_summary.round(1))

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(
    data=geometry,
    x="box_area_pct",
    y="damage_class",
    order=box_size_order,
    showfliers=False,
    color=TEAL,
    ax=ax,
)
ax.set_xscale("log")  # box sizes run from well under 1% to nearly 100% of the photo
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}%"))  # 1%, 10%, 100%
ax.set_xlabel("Box area as a percent of the photo (log scale)")
ax.set_ylabel("")
smallest, largest = box_size_order[0], box_size_order[-1]
ax.set_title(
    f"Median box size: {smallest} {box_size_summary.loc[smallest, 'median_pct']:.1f}% "
    f"of the photo, {largest} {box_size_summary.loc[largest, 'median_pct']:.0f}%"
)
plt.tight_layout()
plt.show()

In [ ]:
# @title Helper: draw a photo with its damage labels { display-mode: "form" }
BOX_COLOR = "#F7B32B"  # yellow boxes and labels
OUTLINE_COLOR = "#53A7D8"  # blue damage outlines


def draw_labeled_photo(ax, photo_path, photo_areas):
    """Show a photo with each damage area's box, damage type, and outline."""
    with Image.open(photo_path) as photo:
        ax.imshow(photo.convert("RGB"))

    for area in photo_areas.itertuples():
        ax.add_patch(
            Rectangle(
                (area.x, area.y),
                area.box_w,
                area.box_h,
                fill=False,
                edgecolor=BOX_COLOR,
                linewidth=1.5,
            )
        )
        ax.text(
            area.x,
            area.y,
            area.damage_class,
            fontsize=8,
            color="black",
            bbox={"facecolor": BOX_COLOR, "alpha": 0.85, "pad": 1},
        )
        # Outlines are stored as flat [x1, y1, x2, y2, ...] lists of points.
        for outline in area.segmentation:
            ax.add_patch(
                Polygon(
                    np.asarray(outline).reshape(-1, 2), facecolor=OUTLINE_COLOR, alpha=0.3
                )
            )
    ax.axis("off")

### Example photos with their labels
One test photo per damage type (the lowest photo id that contains it), so the grid is the same on every run.

In [ ]:
test_geometry = geometry[geometry["split"] == "test"]

example_photo_ids = []
for damage_class in categories.values():
    photo_ids = sorted(
        test_geometry.loc[test_geometry["damage_class"] == damage_class, "image_id"]
    )
    unused_ids = [photo_id for photo_id in photo_ids if photo_id not in example_photo_ids]
    if unused_ids:
        example_photo_ids.append(unused_ids[0])

fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))
for ax in axes.ravel():
    ax.axis("off")

# Up to six examples; any unused panel stays blank, so the lengths can differ.
for ax, photo_id in zip(axes.ravel(), example_photo_ids, strict=False):
    photo_areas = test_geometry[test_geometry["image_id"] == photo_id]
    photo_path = CARDD_IMAGE_DIRS["test"] / photo_areas["file_name"].iloc[0]
    draw_labeled_photo(ax, photo_path, photo_areas)
    damage_types = ", ".join(sorted(photo_areas["damage_class"].unique()))
    ax.set_title(f"Test photo {photo_id}: {damage_types}", fontsize=10)

fig.suptitle("CarDD test photos: damage boxes (yellow) and outlines (blue)")
plt.tight_layout()
plt.show()

### What Part B tells us
- **Size.** 4,000 photos with 8,740 labeled damage areas, split by the authors into 2,816 / 810 / 374 photos.
- **Balance.** The three partitions have nearly the same mix of damage types; no type's share differs by more than a few percentage points between them. The dataset as a whole is lopsided, though: about 11 scratches for every flat tire. Stage 4 reports average precision (AP) for each damage type plus overall mAP, so the rare types can't hide behind the common ones.
- **Size of the damage.** The typical crack box covers about 1% of the photo and the typical shattered-glass box about 68%, so the detector has to find both very small and very large things.
- **Consistency.** All file checks came back clean: no boxes outside their photo, no missing files, and no filename shared between partitions. That does not rule out two different photos of the same car.
- **No clean cars.** Every photo has at least one labeled damage area. To check whether the detector cries wolf on a clean car, we need to add some photos of undamaged cars ourselves.

In [ ]:
# Optional: look for photos that are byte-for-byte identical. This finds exact copies only,
# not crops, re-saved files, or different photos of the same car.
if HASH_CARDD_PIXELS:
    photo_hashes = images.assign(
        sha256=[
            file_sha256(CARDD_IMAGE_DIRS[row.split] / row.file_name)
            for row in images.itertuples()
        ]
    )
    partitions_per_hash = photo_hashes.groupby("sha256")["split"].nunique()
    print(
        f"Identical photo files:                 {photo_hashes['sha256'].duplicated().sum()}"
    )
    print(f"Identical files in more than one split: {(partitions_per_hash > 1).sum()}")
else:
    print("Skipped the identical-photo check (set HASH_CARDD_PIXELS = True to run it).")

## Next steps (Stage 4)
- **Price model:** train on the date-based split, lowercase every text column, keep `condition` as recorded, and compare against recorded MMR's \$1,083 mean absolute error on the same test sales.
- **Damage model:** train a detector on CarDD's official split and report per-class AP and overall mAP (COCO-style, averaged over IoU 0.50 to 0.95).
- **False alarms:** collect a small set of photos of undamaged cars.

## Additional references
- CarDD: https://cardd-ustc.github.io/ ; Wang, Li & Wu (2023), *IEEE Transactions on Intelligent Transportation Systems*.
- DSBA 6165 project guidelines.